# TS Convergence: Automatic Resubmission on a Single Node

Demonstrates `refine_temperature_bracket_with_resubmission` from
`phase_diagram_workflows.free_energies.ts_convergence.single` with a local `SingleNodeExecutor`.

A temperature-scaling (TS) run over a temperature bracket is only trustworthy if its forward and backward sweeps agree.
The disagreement is measured by the **criterion** (`|backward[-1] - forward[0]|` of the energy differences, divided by lambda). If it is
above the tolerance the upper bound is lowered and the bracket is run again. This driver does the whole loop for **one structure**:

1. every bracket is one job; when it ends, that job records its criterion in `bracket_log.csv` and, if the bracket did not converge, **submits the next
   bracket itself**;
2. it stops when a bracket meets the tolerance, when the bracket cannot be narrowed further (`exhausted`), or after `max_iterations` narrowing steps (`limit_reached`);
3. all state is on disk (`bracket_log.csv`, `ts_config.json`, small marker files), never in the executor, so calling it again resumes where it stopped.

The job that submits the next bracket needs to know how to build an executor, and a live executor cannot be sent to another process. So the recipe is passed
as an `ExecutorSpec` (executor class + settings).

With a `SingleNodeExecutor` the submitting job stays alive and waits for the next bracket. With `SlurmClusterExecutor` (or `FluxClusterExecutor`) the job
hands over and exits, so a whole sweep can be submitted once and the notebook closed. Only the `ExecutorSpec` below changes.

The example is Al (Mishin EAM, 500 atoms) on 2 cores and takes a few minutes per bracket. It starts from a deliberately hot bracket, 300-1500 K: far above where the crystal can be switched
reversibly, so the forward and backward sweeps disagree, and the driver narrows the bracket until they agree. The step counts are still short of a production run.

In [ ]:
import os
import shutil
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from ase.build import bulk
from executorlib import SingleNodeExecutor
from lammpsparser import get_potential_by_name

# Make the checkout importable when the package is not installed
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase_diagram_workflows.free_energies.ts_convergence.base import pick_best_converged_bracket
from phase_diagram_workflows.free_energies.ts_convergence.plotting import plot_forward_backward
from phase_diagram_workflows.free_energies.ts_convergence.single import (
    ExecutorSpec,
    current_bracket_resource_dict,
    load_bracket_history,
    refine_temperature_bracket_with_resubmission,
)

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

# lammpsparser looks for the potential library under CONDA_PREFIX, and pylammpsmpi starts LAMMPS through `mpiexec`
# from the environment's bin directory: both are missing when the kernel was started without activating the environment
os.environ.setdefault("CONDA_PREFIX", sys.prefix)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

# the bracket jobs run in worker processes that inherit these: keep TensorFlow start-up notices and deprecation warnings out of the notebook
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"

## 1. Structure, potential and calphy settings

`execution_mode` is `"library"`: LAMMPS runs through pylammpsmpi (the LAMMPS python module) on `queue.cores` MPI ranks, so no `lmp` binary is needed.
All results go to a temporary directory (`ROOT`). It is kept until you delete it: the last cell does that when `CLEAN_UP` is set to `True`.

In [ ]:
ROOT = Path(tempfile.mkdtemp(prefix="ts_resubmission_demo_"))   # nothing is deleted automatically, see the last cell

structure = bulk("Al", cubic=True).repeat(5)   # 500 atoms
potential_df = get_potential_by_name("1999--Mishin-Y--Al--LAMMPS--ipr1").to_frame().transpose()

calphy_parameters = {
    "mode": "ts",
    "pressure": 0,
    "temperature": [300, 1500],           # overwritten for every bracket
    "n_equilibration_steps": 12000,
    "n_switching_steps": 24000,
    "n_print_steps": 0,                    # no trajectory files
    "equilibration_control": "berendsen",
    "md": {"thermostat_damping": 0.5},
    "tolerance": {"spring_constant": 0.01, "pressure": 0.5},
    "queue": {"cores": 2, "scheduler": "local"},   # MPI ranks of the LAMMPS run
    "execution_mode": "library",
    "reference_phase": "solid",
    "file_format": "lammps-data",
}

## 2. Run the whole loop with one call

* `initial_bracket=(300, 1500)`, `step_upper=100`: on non-convergence the upper bound drops by 100 K (1500 -> 1400 -> 1300 ...).
* `tolerance=0.0075`: the criterion a bracket has to get below.
* `max_iterations=3`: at most 3 narrowing steps after the first bracket, so never more than 4 brackets (every bracket is a full job, hence no open-ended loop).
* `resource_dict=current_bracket_resource_dict(...)`: names the executor task after the structure folder and the current bracket. executorlib's default
  name is a hash of the arguments, which do not contain the bracket, so without this a bracket's job would share its name with the previous one and could be
  skipped as "already running" or "already done".

The `SingleNodeExecutor` is only used for this first call; the jobs for the later brackets build their own executor from `EXECUTOR_SPEC`.

In [ ]:
EXECUTOR_SPEC = ExecutorSpec(
    SingleNodeExecutor,
    {"hostname_localhost": True, "cache_directory": str(ROOT / "executorlib_cache")},
)

INITIAL_BRACKET = (300.0, 1500.0)
TOLERANCE = 0.0075
structure_root = str(ROOT / "structure")

executor = EXECUTOR_SPEC.create()
future = executor.submit(
    refine_temperature_bracket_with_resubmission,
    resource_dict=current_bracket_resource_dict(structure_root, INITIAL_BRACKET, EXECUTOR_SPEC),
    input_structure=structure,
    calphy_parameters=calphy_parameters,
    potential_df=potential_df,
    executor_spec=EXECUTOR_SPEC,
    working_directory_root=structure_root,
    initial_bracket=INITIAL_BRACKET,
    tolerance=TOLERANCE,
    step_upper=100.0,
    max_iterations=3,
)
first_job = future.result()    # a single node executor stays alive, so this returns once the whole chain has finished
executor.shutdown(wait=True)
print("first job:", first_job["status"], "| next bracket:", first_job["bracket"])

## 3. What happened

`bracket_log.csv` is a plain table, one row per bracket: the criterion is filled in when the bracket has finished. `pick_best_converged_bracket` returns the widest bracket under the tolerance.

In [ ]:
bracket_log = pd.read_csv(ROOT / "structure" / "bracket_log.csv")
display(bracket_log)

tried_brackets, criteria = load_bracket_history(structure_root)
best = pick_best_converged_bracket(criteria, TOLERANCE)
print("converged bracket:", best if best is not None else "none: every bracket is above the tolerance and max_iterations is used up")

### Criterion against the upper bound of the bracket

In [ ]:
log = bracket_log.sort_values("t_high")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(log["t_high"], log["criterion"], "o-", color="C0")
ax.axhline(TOLERANCE, color="k", ls="--", lw=1, label=f"tolerance ({TOLERANCE:g})")
ax.set_xlabel("upper bound of the bracket [K]")
ax.set_ylabel("criterion")
ax.legend()
plt.tight_layout()
plt.show()

### Forward and backward energy difference in every bracket

`plot_forward_backward` draws one panel per bracket, widest first. The two sweeps of a converged bracket lie on top of each other; a gap between them is dissipation, and the gap at the end of the sweep is what the criterion measures.

In [ ]:
fig, axes = plot_forward_backward(structure_root)
plt.show()

## 4. Calling it again resumes from disk

The same call in a new session (or after a crash) reads `bracket_log.csv` and the marker files. Nothing that is finished is run again: here it just reports where the sweep ended.
Statuses: `converged`, `exhausted`, `limit_reached` (nothing more to do), `resubmitted` (the next bracket was submitted), `running` (a bracket's folder shows recent activity, so it was left alone).

In [ ]:
result = refine_temperature_bracket_with_resubmission(
    input_structure=structure,
    calphy_parameters=calphy_parameters,
    potential_df=potential_df,
    executor_spec=EXECUTOR_SPEC,
    working_directory_root=structure_root,
    initial_bracket=INITIAL_BRACKET,
    tolerance=TOLERANCE,
    step_upper=100.0,
    max_iterations=3,
)
print(result["status"], result["bracket"])

## 5. On a cluster

Only the executor recipe changes. With `SlurmClusterExecutor` every bracket is a SLURM job that submits its successor and exits, so the notebook can be closed right after the first submit:

```python
from executorlib import SlurmClusterExecutor

EXECUTOR_SPEC = ExecutorSpec(
    SlurmClusterExecutor,
    {
        "resource_dict": {"cores": 1, "threads_per_core": 128, "run_time_limit": 5 * 3600},
        "cache_directory": "executorlib_cache",
    },
)
executor = EXECUTOR_SPEC.create()
executor.submit(refine_temperature_bracket_with_resubmission, resource_dict=current_bracket_resource_dict(...), ...)
executor.shutdown(wait=False)    # hand over to SLURM and return
```

Re-running that submit cell is the resume. Before re-running, check `squeue`: a job that is still queued leaves nothing on disk yet.

In [ ]:
CLEAN_UP = False   # set to True to delete the results of this notebook (`ROOT`) once you are done with them
if CLEAN_UP:
    shutil.rmtree(ROOT)